# Transformación de features con Amazon SageMaker Processing y scikit-learn (BYOC)

En este notebook usamos Amazon SageMaker Processing con un container propio (BYOC)
que ejecuta un script de scikit-learn para preprocesar datos de ventas para el entrenamiento de un predictor.

Es necesario ejecutar el siguiente comando para crear la imagen en ECR.  
```bash container/build_and_push.sh sagemaker-processing-byoc```

## Contenidos

1. [Setup](#setup)
1. [Descarga del dataset y carga a S3](#dataset)
1. [Construcción del container](#container)
1. [Script de preprocessing](#script)
1. [Ejecutar el processing job](#run)
1. [Inspeccionar el output](#inspect)

## Setup {#setup}

Especifica el S3 bucket, prefixes y el IAM role para el processing job.

In [21]:
from time import gmtime, strftime
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
timestamp_prefix = strftime("%Y-%m-%d-%H-%M-%S", gmtime())

prefix = "sagemaker/sklearn-preprocess-t6"

if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

input_prefix = prefix + "/input/raw"
input_preprocessed_prefix = prefix + "/input/preprocessed"

print(f"Bucket: {bucket}")
print(f"Input S3 path: s3://{bucket}/{input_prefix}")
print(f"Output S3 path: s3://{bucket}/{input_preprocessed_prefix}")

Bucket: sagemaker-us-east-1-947421917887
Input S3 path: s3://sagemaker-us-east-1-947421917887/sagemaker/sklearn-preprocess-t6/input/raw
Output S3 path: s3://sagemaker-us-east-1-947421917887/sagemaker/sklearn-preprocess-t6/input/preprocessed


## Carga del dataset a S3 {#dataset}


In [13]:
#import boto3

# Ruta local de los datos
data_path = "../data/raw"

#s3 = boto3.client("s3")
#region = sagemaker_session.boto_region_name
#input_data = f"s3://{bucket}/{input_prefix}".format(region)

s3_data_uri = sagemaker_session.upload_data(
    path=data_path,
    bucket=bucket,
    key_prefix=input_prefix
)
print(f" Datos cargados a: {s3_data_uri}\n")

# Verifiacación de datos subidos
response = sagemaker_session.boto_session.client("s3").list_objects_v2(
        Bucket=bucket,
        Prefix=input_prefix
    )
print(f"\n Archivos en S3:")
if "Contents" in response:
    for obj in response["Contents"]:
        print(f"   {obj['Key']}")

 Datos cargados a: s3://sagemaker-us-east-1-947421917887/sagemaker/sklearn-preprocess-t6/input/raw


 Archivos en S3:
   sagemaker/sklearn-preprocess-t6/input/raw/data_en/item_categories_en.csv
   sagemaker/sklearn-preprocess-t6/input/raw/data_en/items_en.csv
   sagemaker/sklearn-preprocess-t6/input/raw/data_en/shops_en.csv
   sagemaker/sklearn-preprocess-t6/input/raw/item_categories.csv
   sagemaker/sklearn-preprocess-t6/input/raw/items.csv
   sagemaker/sklearn-preprocess-t6/input/raw/sales_train.csv
   sagemaker/sklearn-preprocess-t6/input/raw/sample_submission.csv
   sagemaker/sklearn-preprocess-t6/input/raw/shops.csv
   sagemaker/sklearn-preprocess-t6/input/raw/test.csv


### Push a Amazon ECR

In [22]:
import boto3

account_id = boto3.client("sts").get_caller_identity().get("Account")
region = boto3.session.Session().region_name

ecr_repository = "sagemaker-processing-byoc"
tag = ":latest"
uri_suffix = "amazonaws.com"

sklearn_repository_uri = "{}.dkr.ecr.{}.{}/{}".format(
    account_id, region, uri_suffix, ecr_repository + tag
)

## Ejecutar el processing job {#run}

`ScriptProcessor` ejecuta el script en el container BYOC con una sola instancia.
`ProcessingInput` y `ProcessingOutput` le dicen a SageMaker qué datos mover entre
S3 y el container antes y después del job.

| Parámetro | Valor | Impacto |
|-----------|-------|---------|
| `instance_count` | `1` | Una sola instancia — no hay topología scheduler/worker |
| `instance_type` | `ml.m5.large` | 2 vCPUs y 8 GB RAM |
| `command` | `["python3"]` | SageMaker ejecuta `python3 preprocess.py` |
| `ProcessingInput` | `s3://…/raw/` → `/opt/ml/processing/input/` | SageMaker descarga el CSV antes de iniciar el script |
| `ProcessingOutput` | `/opt/ml/processing/output/` → `s3://…/preprocessed/` | SageMaker sube los CSVs de salida al terminar |

In [23]:
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor

sklearn_processor = ScriptProcessor(
    base_job_name="sklearn-preprocessor",
    image_uri=sklearn_repository_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    max_runtime_in_seconds=1200,
)

sklearn_processor.run(
    code="container/preprocess.py",
    inputs=[
        ProcessingInput(
            source=s3_data_uri,
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="prep_data",
            source="/opt/ml/processing/output",
            destination="s3://{}/{}".format(bucket, input_preprocessed_prefix)
        )
    ],
    logs=True,
)

INFO:sagemaker:Creating processing-job with name sklearn-preprocessor-2026-03-17-06-18-46-984


.......2026-03-17 06:19:56,769 - prep - INFO - Logger inicializado. Archivo: /opt/ml/processing/artifacts/logs/prep_20260317_061956.log
2026-03-17 06:19:56,769 - prep - INFO - Iniciando preprocesamiento...
2026-03-17 06:19:57,972 - prep - INFO - Cargado sales_train.csv (rows=2935849, cols=6)
2026-03-17 06:19:58,029 - prep - INFO - Cargado test.csv (rows=214200, cols=3)
2026-03-17 06:19:58,069 - prep - INFO - Cargado items.csv (rows=22170, cols=3)
2026-03-17 06:19:58,070 - prep - INFO - Cargado item_categories.csv (rows=84, cols=2)
2026-03-17 06:19:58,071 - prep - INFO - Cargado shops.csv (rows=60, cols=2)
2026-03-17 06:19:58,100 - prep - INFO - Cargado sample_submission.csv (rows=214200, cols=2)
2026-03-17 06:19:58,915 - prep - INFO - Último date_block_num: 33 | Bloque test: 34
2026-03-17 06:19:59,073 - prep - INFO - Guardado monthly.pkl (rows=1609124)
2026-03-17 06:19:59,073 - prep - INFO - Guardado base.pkl (rows=1823324)
2026-03-17 06:19:59,073 - prep - INFO - Fin prep. Tiempo: 2.30

## Inspeccionar el output {#inspect}

Revisa las primeras filas del dataset transformado para verificar que el preprocessing
fue exitoso.

In [24]:
input_preprocessed_prefix

'sagemaker/sklearn-preprocess-t6/input/preprocessed'

In [25]:
import pandas as pd

monthly_path = "s3://{}/{}/monthly.pkl".format(bucket, input_preprocessed_prefix)
base_path = "s3://{}/{}/base.pkl".format(bucket, input_preprocessed_prefix)

monthly = pd.read_pickle(monthly_path)
base = pd.read_pickle(base_path)

print("Shape:", monthly.shape)
monthly.head()

Shape: (1609124, 6)


,date_block_num,shop_id,item_id,item_cnt_month,avg_price,item_category_id
0,0,0,32,6.0,221.0,40
1,0,0,33,3.0,347.0,37
2,0,0,35,1.0,247.0,40
3,0,0,43,1.0,221.0,40
4,0,0,51,2.0,128.5,57


In [26]:
print("Shape:", base.shape)
base.head()

Shape: (1823324, 6)


,date_block_num,shop_id,item_id,item_cnt_month,avg_price,item_category_id
0,0,0,32,6.0,221.0,40
1,0,0,33,3.0,347.0,37
2,0,0,35,1.0,247.0,40
3,0,0,43,1.0,221.0,40
4,0,0,51,2.0,128.5,57


Los archivos de salida pueden usarse directamente como input para un training job.